# Workshop 2: Análisis de Spotify y Grammys

En este notebook vamos a analizar los datos procesados por nuestro pipeline ETL de Airflow. Los datos ya pasaron por limpieza, validación (Pandera) y el cruce entre Spotify y los ganadores de los Grammys, y ahora están alojados en nuestra base de datos local SQLite.

El objetivo es responder: **¿Qué características tienen los artistas y las canciones ganadoras de Grammys dentro de Spotify?**

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

db_path = os.path.join('..', 'db', 'musica_destino.db')
conn = sqlite3.connect(db_path)

color_grammy = '#2a78d6'   
color_sin_grammy = '#eb6834' 
colores = [color_grammy, color_sin_grammy]

print("Conexión exitosa a la base de datos:", db_path)

## 1. Popularidad Promedio: ¿Tener un Grammy te hace más popular en Spotify?
Vamos a comparar la métrica `popularity` que nos da Spotify para ambos grupos de canciones.

In [ ]:
query_pop = """
    SELECT CASE WHEN has_grammy = 1 THEN 'Con Grammy' ELSE 'Sin Grammy' END AS grupo,
           AVG(popularity) AS popularidad_promedio,
           COUNT(*) AS canciones
    FROM spotify_grammys 
    GROUP BY grupo 
    ORDER BY grupo
"""
df_pop = pd.read_sql(query_pop, conn)

plt.figure(figsize=(8, 4))
plt.bar(df_pop['grupo'], df_pop['popularidad_promedio'], color=colores)
plt.title('1. Popularidad Promedio: con vs. sin Grammy')
plt.ylabel('Popularidad (0-100)')
plt.show()

**Análisis:** Curiosamente, la popularidad promedio es prácticamente idéntica (~33 puntos). Esto rompe nuestro mito inicial: ganar un Grammy no significa automáticamente que tus canciones se escuchen masivamente en la era del streaming (Spotify). Hay muchas canciones virales de artistas sin Grammy que nivelan este promedio.

## 2. Top 10 Artistas con más Grammys en Spotify
Queremos ver quiénes son los pesos pesados de la industria musical que logramos cruzar en nuestro dataset.

In [ ]:
query_top = """
    SELECT main_artist, MAX(grammy_wins) AS grammys
    FROM spotify_grammys 
    WHERE has_grammy = 1
    GROUP BY artist_key 
    ORDER BY grammys DESC 
    LIMIT 10
"""
df_top = pd.read_sql(query_top, conn)
df_top = df_top.iloc[::-1] 

plt.figure(figsize=(8, 5))
plt.barh(df_top['main_artist'], df_top['grammys'], color=color_grammy)
plt.title('2. Top 10 Artistas con más Grammys en Spotify')
plt.xlabel('Cantidad de Grammys ganados')
plt.show()

**Análisis:** Alison Krauss, Ray Charles, Aretha Franklin y Beyoncé lideran. Es lógico porque son artistas con carreras larguísimas y multipremiadas. 

## 3. Géneros con mayor proporción de premiados
¿Qué géneros musicales concentran a la mayor cantidad de artistas reconocidos por la academia?

In [ ]:
query_gen = """
    SELECT track_genre, ROUND(100.0 * SUM(has_grammy) / COUNT(*), 1) AS pct_con_grammy
    FROM spotify_grammys 
    GROUP BY track_genre 
    HAVING COUNT(*) >= 50
    ORDER BY pct_con_grammy DESC 
    LIMIT 10
"""
df_gen = pd.read_sql(query_gen, conn)
df_gen = df_gen.iloc[::-1]

plt.figure(figsize=(8, 5))
plt.barh(df_gen['track_genre'], df_gen['pct_con_grammy'], color=color_grammy)
plt.title('3. Géneros con mayor % de canciones de artistas con Grammy')
plt.xlabel('% de canciones del género')
plt.show()

**Análisis:** El 'honky-tonk' (un subgénero del country) y el 'rock' dominan aquí. Esto tiene sentido ya que históricamente los Grammys han premiado mucho más a los géneros tradicionales estadounidenses en comparación con géneros más modernos o electrónicos.

## 4. Perfil de Audio Promedio
Spotify nos da métricas súper interesantes de la estructura de la canción (si es bailable, enérgica, acústica, etc.). Vamos a ver cómo suenan los ganadores vs los no ganadores.

In [ ]:
query_audio = """
    SELECT CASE WHEN has_grammy = 1 THEN 'Con Grammy' ELSE 'Sin Grammy' END AS grupo,
           AVG(danceability) AS danceability, 
           AVG(energy) AS energy,
           AVG(valence) AS valence, 
           AVG(acousticness) AS acousticness,
           AVG(speechiness) AS speechiness
    FROM spotify_grammys 
    GROUP BY grupo 
    ORDER BY grupo
"""
df_audio = pd.read_sql(query_audio, conn)

df_audio.set_index('grupo').T.plot(kind='bar', figsize=(10, 5), color=colores)
plt.title('4. Perfil de Audio Promedio (0 a 1)')
plt.xticks(rotation=0)
plt.show()

**Análisis:** Aquí sí hay una diferencia notable. Las canciones de artistas ganadores tienden a ser más acústicas (`acousticness`) y un poco menos enérgicas (`energy`). La música comercial sin premios tiende a ser más ruidosa y producida electrónicamente.

## 5. Década del primer reconocimiento
Veamos de qué época son los artistas que quedaron en nuestro dataset final después del merge.

In [ ]:
query_dec = """
    SELECT (CAST(first_grammy_year AS INTEGER) / 10) * 10 AS decada,
           COUNT(DISTINCT artist_key) AS artistas
    FROM spotify_grammys 
    WHERE has_grammy = 1 
    GROUP BY decada 
    ORDER BY decada
"""
df_dec = pd.read_sql(query_dec, conn)

plt.figure(figsize=(8, 4))
plt.bar(df_dec['decada'].astype(int).astype(str) + "s", df_dec['artistas'], color=color_grammy)
plt.title('5. Artistas con Grammy según la década de su primer premio')
plt.ylabel('Número de artistas distintos')
plt.show()

**Análisis:** La mayoría de los artistas premiados en nuestro cruce ganaron su primer Grammy entre la década de los 90s y los 2010s. Esto es esperado por un sesgo natural: el catálogo de Spotify contiene mucha más música contemporánea que música de los años 50s.

## 6. Proporción del Catálogo: Canciones Totales
Por último, para dar contexto, ¿cuántas canciones procesadas pertenecen a cada grupo(con y sin grammy)?

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_pop['grupo'], df_pop['canciones'], color=colores)
plt.title('6. Número de canciones según su artista')
plt.ylabel('Total de canciones en Spotify')

# Agregamos los números exactos encima de las barras para mayor claridad
for i, v in enumerate(df_pop['canciones']):
    plt.text(i, v + 1000, f"{v:,}", ha='center')

plt.show()

# Cerramos la conexión a la base de datos
conn.close()

**Análisis Final:** De las casi 90,000 canciones limpias que entraron al modelo, solo ~7,400 pertenecen a artistas con al menos un Grammy (alrededor del 8.3%). Esto nos recuerda que los ganadores de un Grammy son una minoría súper exclusiva dentro del enorme mar de artistas que existen en Spotify. El ETL funcionó perfectamente para segmentar esta élite y compararla con el resto de la industria.